# Tips Analysis FHVHV

Data range from 2020 to 2025 (excluding 2025 December)

Run it with *.csv from `TABLES_DIR` if it's in your directory.

In [ ]:
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as f
import findspark
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np
import os
from pathlib import Path

In [ ]:
projectRoot = Path.cwd().parents[2]

CLEAN_DATA = projectRoot / "data" / "clean" / "clean_tlc_trip_record" / "fhvhv"
# csv files of plot data
TABLES_DIR = projectRoot / "data" / "plotTables" / "tips" / "fhvhv"
os.makedirs(TABLES_DIR, exist_ok=True)

HTML_DIR = projectRoot / "data" / "html" / "tips" / "fhvhv"
os.makedirs(HTML_DIR, exist_ok=True)

In [ ]:
def save_table(df:pd.DataFrame, name:str):
    path = os.path.join(str(TABLES_DIR), f"{name}.csv")
    df.to_csv(path, index=False)

def load_csv(path:Path) -> pd.DataFrame:
    return pd.read_csv(path)

def make_date_col(df):
    """
    Build a pandas datetime column from pickup_year + pickup_month.

    2020 01 -> 2020-01-01
    """
    df["date"] = pd.to_datetime(
        df["pickup_year"].astype(str) + "-" +
        df["pickup_month"].astype(str) + "-01"
    )
    return df

def add_temporal_columns(df: DataFrame) -> DataFrame:
    """
    Derive useful temporal features.
    """
    df = (
        df
        .withColumn("pickup_hour", f.hour("pickup_datetime"))
        .withColumn("pickup_day_of_week", f.dayofweek("pickup_datetime"))
        .withColumn("pickup_day_name", f.date_format("pickup_datetime", "EEEE"))
        .withColumn("pickup_is_weekend", f.dayofweek("pickup_datetime").isin([1, 7]))
        .withColumn("time_of_day",
            f.when((f.col("pickup_hour") >= 6)  & (f.col("pickup_hour") < 10), "morning_rush")
             .when((f.col("pickup_hour") >= 10) & (f.col("pickup_hour") < 16), "midday")
             .when((f.col("pickup_hour") >= 16) & (f.col("pickup_hour") < 20), "evening_rush")
             .when((f.col("pickup_hour") >= 20) & (f.col("pickup_hour") < 24), "night")
             .otherwise("late_night")
        )
    )
    return df

In [ ]:
LAYOUT_DEFAULTS = dict(
    template="plotly_white",
    font=dict(family="Inter, sans-serif", size=12),
    hovermode="x unified",
    margin=dict(l=60, r=30, t=60, b=50),
)

DAY_ORDER = ["Monday", "Tuesday", "Wednesday", "Thursday",
             "Friday", "Saturday", "Sunday"]

COMPANY_COLORS = {
    "Uber": "#000000", "Lyft": "#FF00BF",
    "Via": "#00C48C",  "Juno": "#FFA500",
}

# Spark Init

## No need to run Spark if you run the code from *.csv

`NOTE`:
Change the configs according to your device if you wanna run it (if you don't have the plot tables already in your device).

`If we want to run this code on the cloud,
we will need to change the configs too.`

In [ ]:
findspark.init()

spark = (SparkSession.builder
        .master("local[*]")  
        .config("spark.driver.memory", "24g")
        .config("spark.executor.memory", "24g")
        .config("spark.sql.shuffle.partitions", "100")
        .config("spark.sql.adaptive.advisoryPartitionSizeInBytes", "64m")
        .config("spark.driver.maxResultSize", "1g")
        .config("spark.sql.session.timeZone", "America/New_York")
        .config("spark.sql.timestampType", "TIMESTAMP_NTZ")
        # Network binding fixes for macOS
        .config("spark.driver.host", "127.0.0.1")
        .config("spark.driver.bindAddress", "127.0.0.1")
        .getOrCreate())

In [ ]:
df = spark.read.parquet(str(CLEAN_DATA))

df = add_temporal_columns(df)

# Tips per borough

In [ ]:
def temp_01_heatmap_borough_tips_simple(df: DataFrame, table_path: Path = None):
    
    if not table_path:
        heat = (
            df.groupBy("pickup_borough", "dropoff_borough")
            .agg(
                f.avg("tips").alias("avg_tip"),
                f.count("*").alias("trips")
            )
            .filter(f.col("trips") > 10)
            .toPandas()
        )
        save_table(heat, "temp_01_heatmap_borough_tips_simple")
    else:
        heat = pd.read_csv(table_path)
    
    pivot = heat.pivot_table(
        index="pickup_borough",
        columns="dropoff_borough",
        values="avg_tip",
        aggfunc="mean"
    )
    
    pivot = pivot.fillna(0)
    
    fig = go.Figure(data=go.Heatmap(
        z=pivot.values,
        x=pivot.columns.tolist(),
        y=pivot.index.tolist(),
        colorscale="RdBu_r",
        text=np.round(pivot.values, 2),
        texttemplate="$%{text:.2f}",
        hovertemplate="Pickup: %{y}<br>Dropoff: %{x}<br>Tip: $%{z:.2f}<extra></extra>"
    ))
    
    fig.update_layout(
        title="Average Tips per Borough (Pickup → Dropoff)",
        xaxis_title="Dropoff Borough",
        yaxis_title="Pickup Borough",
        height=500,
        width=700, **LAYOUT_DEFAULTS
    )
    
    fig.show()

In [ ]:
temp_01_heatmap_borough_tips_simple(df, None)

#temp_01_heatmap_borough_tips_simple(None, TABLES_DIR / "temp_01_heatmap_borough_tips_simple.csv")

# Tips Temporal

In [ ]:
def temp_02_tips_temporal(df: DataFrame, table_path: Path):
    """
    Monthly tips trend: avg tip amount, % of trips with a tip, avg tip rate (% of base fare).
    """
    if not table_path:
        tips = (
            df.groupBy("pickup_year", "pickup_month")
            .agg(
                f.round(f.avg("tips"), 2).alias("avg_tip"),
                f.round(
                    f.sum(f.when(f.col("tips") > 0, 1).otherwise(0)) / f.count("*") * 100, 1
                ).alias("pct_tipped"),
                f.round(
                    f.avg(
                        f.when(f.col("base_passenger_fare") > 0,
                               f.col("tips") / f.col("base_passenger_fare") * 100)
                    ), 2
                ).alias("avg_tip_rate"),
            )
            .orderBy("pickup_year", "pickup_month")
            .toPandas()
        )
        tips = make_date_col(tips)
        save_table(tips, "temp_02_tips_temporal")
    else:
        tips = load_csv(table_path)

    fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=("Avg Tip Amount", "% Trips with Tip", "Avg Tip Rate (% of base fare)"),
    )

    traces = [
        ("avg_tip",      "#61c1a2", "USD ($)"),
        ("pct_tipped",   "#8093c1", "% Trips"),
        ("avg_tip_rate", "#f28459", "% of Base Fare"),
    ]
    for col_idx, (col, color, ylabel) in enumerate(traces, start=1):
        fig.add_trace(go.Scatter(
            x=tips["date"], y=tips[col],
            mode="lines+markers", name=ylabel,
            line=dict(color=color, width=2),
            marker=dict(size=5),
            hovertemplate="%{x|%b %Y}: %{y:.2f}<extra></extra>",
        ), row=1, col=col_idx)
        fig.update_yaxes(title_text=ylabel, row=1, col=col_idx)

    fig.update_layout(
        title="FHVHV - Tips Monthly Trend (2020 - 2025)",
        showlegend=False, height=420, **LAYOUT_DEFAULTS,
    )
    fig.show()

In [ ]:
temp_02_tips_temporal(df, None)

# temp_02_tips_temporal(None, TABLES_DIR / "temp_02_tips_temporal.csv")

# Trips Temporal Borough 

In [ ]:
def temp_03_tips_borough(df: DataFrame, path_file: Path):
    """All pickup boroughs: monthly trip trends."""
    if not path_file:
        boro = (
            df.groupBy("pickup_year", "pickup_month", "pickup_borough")
            .agg(f.round(f.avg("tips"), 2).alias("avg_tip"))
            .toPandas()
        )
        boro = make_date_col(boro)
        
        save_table(boro, "temp_03_tips_borough")
    else:
        boro = load_csv(path_file)
    all_boroughs = boro["pickup_borough"].unique()
    borough_colors = px.colors.qualitative.Set2

    fig = go.Figure()
    for i, borough in enumerate(all_boroughs):
        sub = boro[boro["pickup_borough"] == borough].sort_values("date")
        fig.add_trace(go.Scatter(
            x=sub["date"], y=sub["avg_tip"],
            mode="lines", name=borough,
            line=dict(color=borough_colors[i % len(borough_colors)], width=2),
            hovertemplate=f"{borough}<br>" + "%{x|%b %Y}: %{y:,.0f}<extra></extra>",
        ))

    fig.update_layout(
        title="FHVHV - Monthly Avg Tips by Borough (2020- 2025)",
        yaxis_title="Avg Tips (USD $)", xaxis_title="",
        height=500, **LAYOUT_DEFAULTS,
    )
    fig.show()
    fig.write_html(HTML_DIR / f"temp_03_tips_borough.html")

In [ ]:
temp_03_tips_borough(df, None)

# temp_03_tips_borough(None, TABLES_DIR / "temp_03_tips_borough.csv")

# Tips Distribution in NYC

In [ ]:
def temp_04_tips_distribution_nyc(df: DataFrame, path_file: Path):
    """Tips as percentage of total amount distribution - with bins in Spark."""
    if not path_file:
        tips_with_ranges = (
            df.select(
                ((f.col('tips') / f.col('base_passenger_fare')) * 100).alias('tip_percentage')
            )
            # .filter(f.col('base_passenger_fare') > 0)
            # .filter(f.col('tip_percentage').between(0, 25))  # Filtrar hasta 25%
            .withColumn('percentage_range',
                f.when(f.col('tip_percentage') < 5, "0-5%")
                .when(f.col('tip_percentage') < 10, "5-10%")
                .when(f.col('tip_percentage') < 15, "10-15%")
                .when(f.col('tip_percentage') < 20, "15-20%")
                .when(f.col('tip_percentage') < 25, "20-25%")
                .when(f.col('tip_percentage') < 30, "25-30%")
                .otherwise("30-50%")
            )
        )
        
        # Crear columna con orden numérico
        tips_with_order = tips_with_ranges.withColumn('range_order',
            f.when(f.col('percentage_range') == "0-5%", 1)
            .when(f.col('percentage_range') == "5-10%", 2)
            .when(f.col('percentage_range') == "10-15%", 3)
            .when(f.col('percentage_range') == "15-20%", 4)
            .when(f.col('percentage_range') == "20-25%", 5)
            .when(f.col('percentage_range') == "25-30%", 6)
            .when(f.col('percentage_range') == "30-50%", 7)
        )
        
        range_counts = (
            tips_with_order.groupBy('percentage_range', 'range_order')
            .count()
            .orderBy('range_order')  # Ordenar por orden numérico
            .toPandas()
        )
        
        save_table(range_counts, "temp_04_tips_distribution_nyc")
    else:
        range_counts = load_csv(path_file)
    
    # Crear gráfica de barras con los rangos
    fig = go.Figure()
    
    fig.add_trace(go.Bar(
        x=range_counts['percentage_range'],
        y=range_counts['count'],
        marker_color='rgb(99, 110, 250)',
        opacity=0.8,
        name="Tips %",
        hovertemplate="Range: %{x}<br>Trips: %{y:,.0f}<br>Percentage: %{customdata:.1f}%<extra></extra>",
        customdata=(range_counts['count'] / range_counts['count'].sum() * 100)
    ))
    
    # Destacar rango típico
    fig.update_layout(
        title="FHVHV - Tip Percentage Distribution by Range (2020-2025)",
        xaxis_title="Tip Percentage Range",
        yaxis_title="Number of Trips",
        height=600,
        showlegend=False,
        **LAYOUT_DEFAULTS
    )
    
    fig.update_traces(marker_color=[
        "#7ADAA8" if r == '15-20%' else "#7F8DE2" 
        for r in range_counts['percentage_range']
    ])
    
    fig.show()
    fig.write_html(HTML_DIR / "temp_04_tips_distribution_nyc.html")

In [ ]:
temp_04_tips_distribution_nyc(df, None)

#temp_04_tips_distribution_nyc(None, TABLES_DIR / "temp_04_tips_distribution_nyc.csv")

# Tips Distribution in Manhattan and Queens

In [ ]:
def temp_05_tips_distribution_all_boroughs(df: DataFrame, path_file: Path):
    """Tips distribution comparison - All boroughs with percentages."""
    if not path_file:
        tips_with_ranges = (
            df.select(
                'pickup_borough',
                ((f.col('tips') / f.col('base_passenger_fare')) * 100).alias('tip_percentage')
            )
            .withColumn('percentage_range',
                f.when(f.col('tip_percentage') < 5, "0-5%")
                .when(f.col('tip_percentage') < 10, "5-10%")
                .when(f.col('tip_percentage') < 15, "10-15%")
                .when(f.col('tip_percentage') < 20, "15-20%")
                .when(f.col('tip_percentage') < 25, "20-25%")
                .when(f.col('tip_percentage') < 30, "25-30%")
                .otherwise("30-50%")
            )
        )
        
        range_counts = (
            tips_with_ranges.groupBy('pickup_borough', 'percentage_range')
            .count()
            .toPandas()
        )
        
        borough_totals = range_counts.groupby('pickup_borough')['count'].sum().reset_index()
        borough_totals = borough_totals.rename(columns={'count': 'total_trips'})
        range_counts = range_counts.merge(borough_totals, on='pickup_borough')
        range_counts['percentage_of_borough'] = (range_counts['count'] / range_counts['total_trips']) * 100
        
        order = ["0-5%", "5-10%", "10-15%", "15-20%", "20-25%", "25-30%", "30-50%"]
        range_counts['percentage_range'] = pd.Categorical(
            range_counts['percentage_range'], 
            categories=order, 
            ordered=True
        )
        range_counts = range_counts.sort_values(['pickup_borough', 'percentage_range'])
        
        save_table(range_counts, "temp_05_tips_distribution_all_boroughs")
    else:
        range_counts = load_csv(path_file)
    
    boroughs = sorted(range_counts['pickup_borough'].unique())
    
    # Paleta de colores en hexadecimal
    color_palette = [
        "#7F8DE2",
        "#FF978A",
        "#7ADAA8",
        "#FFD28E",
        "#CC8EF0",
        "#FFB4D9",
    ]

    fig = go.Figure()
    
    for i, borough in enumerate(boroughs):
        borough_data = range_counts[range_counts['pickup_borough'] == borough]
        
        if not borough_data.empty:
            fig.add_trace(go.Bar(
                name=borough,
                x=borough_data['percentage_range'],
                y=borough_data['percentage_of_borough'],
                marker_color=color_palette[i % len(color_palette)],
                opacity=0.8,
                hovertemplate=f"{borough}<br>Range: %{{x}}<br>% of trips: %{{y:.1f}}%<br>Trips: %{{customdata:,.0f}}<extra></extra>",
                customdata=borough_data['count']
            ))
    
    fig.update_layout(
        title="FHVHV - Tip Percentage Distribution: All Boroughs (2020-2025)<br><sup>Percentage of trips by borough</sup>",
        xaxis_title="Tip Percentage Range",
        yaxis_title="Percentage of Trips (%)",
        barmode='group',
        height=700, 
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="center",
            x=0.5
        ),
        **LAYOUT_DEFAULTS
    )
    
    fig.show()
    fig.write_html(HTML_DIR / "temp_05_tips_distribution_all_boroughs.html")

In [ ]:
temp_05_tips_distribution_all_boroughs(df, None)

#temp_05_tips_distribution_all_boroughs(None, TABLES_DIR / "temp_05_tips_distribution_all_boroughs.csv")

# Correlation

In [ ]:
def temp_06_correlation_matrix(df: DataFrame, path_file: Path):
    """Correlation matrix for numeric features."""
    if not path_file:
        numeric_cols = [
            'tips', 
            'base_passenger_fare', 
            'trip_miles',
            'trip_time',
            'wait_time'
        ]
        
        available_cols = [col for col in numeric_cols if col in df.columns]
        
        from pyspark.ml.feature import VectorAssembler
        from pyspark.ml.stat import Correlation
        
        assembler = VectorAssembler(inputCols=available_cols, outputCol="features")
        df_vector = assembler.transform(df.select(*available_cols).na.drop())
        
        corr_matrix = Correlation.corr(df_vector, "features").head()[0]
        corr_array = corr_matrix.toArray()
        
        corr_df = pd.DataFrame(
            corr_array,
            columns=available_cols,
            index=available_cols
        )
        
        save_table(corr_df, "temp_06_correlation_matrix")
    else:
        corr_df = load_csv(path_file)
    
    fig = go.Figure(data=go.Heatmap(
        z=corr_df.values,
        x=corr_df.columns,
        y=corr_df.index,
        colorscale='RdBu_r',
        zmin=-1, zmax=1,
        text=corr_df.values.round(3),
        texttemplate='%{text}',
        textfont={"size": 12},
        hoverongaps=False,
        hovertemplate='%{x} vs %{y}<br>Correlation: %{z:.3f}<extra></extra>'
    ))
    
    fig.update_layout(
        title="Correlation Matrix",
        xaxis_title="Variables",
        yaxis_title="Variables",
        height=600,
        width=700,
        **LAYOUT_DEFAULTS
    )
    
    fig.show()
    fig.write_html(HTML_DIR / "temp_06_correlation_matrix.html")

In [ ]:
temp_06_correlation_matrix(df, None)

#temp_06_correlation_matrix(None, TABLES_DIR / "temp_06_correlation_matrix.csv")

# Tips by Company

In [ ]:
def temp_07_tips_by_company(df: DataFrame, table_path: Path):
    """
    Monthly avg tip and % of trips tipped, broken down by company.
    """
    if not table_path:
        co = (
            df.groupBy("pickup_year", "pickup_month", "company")
            .agg(
                f.round(f.avg("tips"), 2).alias("avg_tip"),
                f.round(
                    f.sum(f.when(f.col("tips") > 0, 1).otherwise(0)) / f.count("*") * 100, 1
                ).alias("pct_tipped"),
            )
            .toPandas()
        )
        co = make_date_col(co)
        save_table(co, "temp_07_tips_by_company")
    else:
        co = load_csv(table_path)

    fig = make_subplots(
        rows=1, cols=2, shared_xaxes=True,
        subplot_titles=("Avg Tip by Company", "% Trips Tipped by Company"),
    )

    for company in ["Uber", "Lyft", "Via", "Juno"]:
        sub = co[co["company"] == company].sort_values("date")
        if sub.empty:
            continue
        c = COMPANY_COLORS[company]
        fig.add_trace(go.Scatter(
            x=sub["date"], y=sub["avg_tip"],
            mode="lines", name=company, legendgroup=company,
            line=dict(color=c, width=2),
            hovertemplate=f"{company}<br>" + "%{x|%b %Y}: $%{y:.2f}<extra></extra>",
        ), row=1, col=1)
        fig.add_trace(go.Scatter(
            x=sub["date"], y=sub["pct_tipped"],
            mode="lines", name=company, legendgroup=company, showlegend=False,
            line=dict(color=c, width=2),
            hovertemplate=f"{company}<br>" + "%{x|%b %Y}: %{y:.1f}%<extra></extra>",
        ), row=1, col=2)

    fig.update_yaxes(title_text="USD ($)", row=1, col=1)
    fig.update_yaxes(title_text="% Trips", row=1, col=2)

    fig.update_layout(
        title="FHVHV - Tips by Company (2020 - 2025)",
        height=450, **LAYOUT_DEFAULTS,
    )
    fig.show()

In [ ]:
temp_07_tips_by_company(df, None)

#temp_07_tips_by_company(None, TABLES_DIR / "temp_07_tips_by_company.csv")

# Stop Spark

In [ ]:
spark.stop()